# Corpus v1: one-epoch Gemma 3 4B QLoRA pilot
Import as a **new private notebook**. Attach the private Dataset with `curated.jsonl`, **corpus_v1_baseline version 4 output**, and **corpus_v1_smoke version 1 output**. Use **Add Input → Notebook Output Files** for the two notebook outputs. Enable **HF_TOKEN** in Secrets and internet; select **GPU T4 x2**. Only the first T4 is used.

This bounded pilot trains a **fresh rank-8 language q/v LoRA adapter** on base Gemma 3 4B for **exactly one epoch over 247 training pairs**. It does not resume the smoke adapter. Corpus, baseline and successful smoke hashes are checked before loading weights. NF4/FP32, learning rate 5e-5, microbatch 1, accumulation 2, seed 42, limit 1024 without truncation, no packing. Each pair is seen once in shuffled order: **124 updates**, with the last update correctly normalized for its single microbatch. No automatic second epoch or extended training is included.

Completion-only labels are verified for all 247 training and 21 validation pairs. All 21 validation losses are measured before/after training and after adapter reload. The epoch adapter is saved and reloaded before greedy validation generation, using the exact stored prompts and 512-token budget from the frozen base comparator. No test pair is tokenized, trained, generated or scored here.

Outputs under `/kaggle/working/corpus_v1_pilot/`: `pilot_status.json`, `evaluation.json`, `comparison.json`, `validation_generations.jsonl`, `adapter/`, `pilot_manifest.json`, label audit and training logs. `claim_style_review.md` varies A/B order, `comparison_key.json` contains the mapping, and `review_scores.jsonl` has blank separate claim/style fields. Masking labels is not independent blinded evaluation. Copy/length statistics are descriptive only.

Updates and generations are saved incrementally; a failed run stays incomplete. Heartbeats print every 30 seconds during slow operations. Use a fresh session. Based on the smoke timings, budget roughly **25–40 minutes**; actual runtime varies with pair lengths, downloads and generation lengths. A lower loss is not proof of improved claims or style.


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"


In [ ]:
%pip install -q -U "transformers==5.20.1" "peft==0.21.2" "accelerate==1.15.0" "bitsandbytes==0.50.2" "huggingface-hub==1.33.0" "sentencepiece" "Pillow"


In [ ]:
%%writefile /kaggle/working/corpus_baseline.py
"""Frozen multi-paper v1 instruction, identical to the curated data export."""

FORMAT = "fama_multi_paper_v1"
INSTRUCTION = (
    "Rewrite the passage below in the formal academic economics style of Eugene Fama "
    "and his coauthors. Preserve every substantive claim, actor, attribution, "
    "condition, qualification, uncertainty, negation, quantity, comparison, and "
    "causal relationship. Do not add or omit claims or strengthen tentative claims. "
    "Return only one coherent paragraph, without headings, Markdown, explanations, "
    "or commentary."
)


def make_prompt(text):
    return f"{INSTRUCTION}\n\nPassage:\n{text}"


"""Token-length preflight and inference-only 4B baseline for corpus v1."""

import argparse
import collections
import hashlib
import importlib.metadata
import json
import os
import re
import threading
import time
from pathlib import Path


MODEL_ID = "google/gemma-3-4b-it"
DATASET_SHA256 = "c24e2d708884877ce61ea212c4d6e442261728be499ff2d15ee7826887a4299f"
COUNTS = {"train": 247, "validation": 21, "test": 45}
PAPERS = {"F1965": ("train", 30), "F1980": ("train", 6),
          "FF2000": ("train", 40), "FJ1983": ("train", 50),
          "ME1997": ("train", 60), "NL2003": ("train", 40), "VL2018": ("train", 21),
          "FF1997": ("validation", 21), "FF2014": ("test", 45)}
CONTEXT_BUDGET = 4096
IMPLEMENTATION = "corpus_baseline_v4_actual_device_check"


def validate_model_placement(model):
    """Check actual tensors; hf_device_map is optional loader metadata."""
    device_map = getattr(model, "hf_device_map", None)
    if device_map is not None and (not isinstance(device_map, dict) or not device_map
            or any(str(device) not in ("0", "cuda:0") for device in device_map.values())):
        raise PreflightError("model_device_map_not_first_gpu")
    counts = {"parameters": 0, "buffers": 0}
    for kind, tensors in (("parameters", model.parameters()), ("buffers", model.buffers())):
        for tensor in tensors:
            if tensor.device.type != "cuda" or tensor.device.index != 0:
                raise PreflightError("model_tensor_not_first_gpu")
            counts[kind] += 1
    if not counts["parameters"]:
        raise PreflightError("model_has_no_parameters")
    embedding_device = model.get_input_embeddings().weight.device
    if embedding_device.type != "cuda" or embedding_device.index != 0:
        raise PreflightError("model_embedding_not_first_gpu")
    return {"device": "cuda:0", "device_map_present": device_map is not None,
            "parameter_tensors_checked": counts["parameters"], "buffer_tensors_checked": counts["buffers"]}


def initialize_cuda(torch):
    """Initialize the allocator before resetting its counters in a fresh process."""
    torch.cuda.init()
    torch.cuda.set_device(0)
    torch.manual_seed(42)
    torch.cuda.reset_peak_memory_stats(0)


def safe_failure_details(exc):
    """Retain stack locations, never exception messages, source lines or locals.

    Remote errors may include authenticated URLs. Only a closed set of labels,
    numeric line numbers and local/package stack locations may leave this helper.
    """
    frames = []
    tb = exc.__traceback__
    while tb is not None:
        filename = tb.tb_frame.f_code.co_filename.replace("\\", "/")
        category = None
        if filename == __file__:
            category = "corpus_baseline"
        else:
            for package in ("torch", "transformers", "bitsandbytes", "accelerate", "huggingface_hub"):
                if "/" + package + "/" in filename:
                    category = package
                    break
        if category:
            frames.append({"component": category, "line": tb.tb_lineno})
        tb = tb.tb_next
    # Compare internally; never serialize even a redacted remote message.
    text = str(exc)
    code = "unclassified_runtime_failure"
    for marker, label in (
        ("invalid argument to reset_peak_memory_stats", "cuda_allocator_not_initialized"),
        ("CUDA out of memory", "cuda_out_of_memory"),
        ("no kernel image is available", "cuda_kernel_not_supported"),
        ("CUDA driver version is insufficient", "cuda_driver_incompatible"),
    ):
        if marker in text:
            code = label
            break
    if "Invalid device argument" in text and "did you call init?" in text:
        code = "cuda_allocator_not_initialized"
    return {"error_code": code, "stack_locations": frames}


class PreflightError(ValueError):
    """A known local failure code, with no remote exception text or credentials."""
    def __init__(self, code):
        self.code = code
        super().__init__(code)


def load_pinned_config(api, config_class, token, progress=None):
    """Resolve a public Hub SHA, then load exactly that configuration revision.

    Transformers 5.20 discards the old private config._commit_hash attribute.
    Config, processor and weights must instead receive the same public revision.
    """
    info = api.model_info(MODEL_ID, revision="main", timeout=30)
    revision = getattr(info, "sha", None)
    if not isinstance(revision, str) or not re.fullmatch(r"[0-9a-f]{40}", revision):
        raise PreflightError("hub_revision_missing_or_invalid")
    if progress:
        progress("loading_model_config", model_id=MODEL_ID, model_revision=revision)
    config = config_class.from_pretrained(MODEL_ID, revision=revision, token=token)
    return config, revision


def digest(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def write_json(path, value):
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    temporary.replace(path)


def validate_rows(rows):
    if len(rows) != 313 or len({r["id"] for r in rows}) != 313:
        raise ValueError("Expected 313 unique corpus-v1 rows")
    if dict(collections.Counter(r["split"] for r in rows)) != COUNTS:
        raise ValueError("Corpus split counts changed")
    if dict(collections.Counter(r["paper_key"] for r in rows)) != {p: n for p, (_, n) in PAPERS.items()}:
        raise ValueError("Corpus paper counts changed")
    for row in rows:
        if (row["split"] != PAPERS[row["paper_key"]][0] or row["prompt_format"] != FORMAT
                or row["prompt"] != make_prompt(row["plain_english"])
                or row["completion"] != row["original_paragraph"]
                or not row["plain_english"].strip() or not row["completion"].strip()):
            raise ValueError("Changed prompt format, paper split, or source/target alignment")
    return {split: [r for r in rows if r["split"] == split] for split in COUNTS}


def load_corpus(path):
    raw = Path(path).read_bytes()
    if hashlib.sha256(raw).hexdigest() != DATASET_SHA256:
        raise ValueError("Attach the exact current data/fama_corpus_v1/curated.jsonl; SHA-256 mismatch")
    rows = [json.loads(line) for line in raw.decode("utf-8").splitlines() if line.strip()]
    return rows, validate_rows(rows)


def load_hf_token():
    if Path("/kaggle").exists():
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            raise RuntimeError("Enable the Kaggle Secret named HF_TOKEN for this notebook") from None
    else:
        token = os.environ.get("HF_TOKEN")
    if not isinstance(token, str) or not token.strip():
        raise RuntimeError("HF_TOKEN is unavailable; configure it outside notebook code and the repository")
    return token


def user_message(row):
    return {"role": "user", "content": [{"type": "text", "text": row["prompt"]}]}


def input_ids(batch):
    value = batch["input_ids"]
    value = value.tolist() if hasattr(value, "tolist") else value
    return value[0] if value and isinstance(value[0], list) else value


def prepare_pair(processor, row):
    """Verify an exact completion boundary, without truncation or exposing token text."""
    user = user_message(row)
    assistant = {"role": "assistant", "content": [{"type": "text", "text": row["completion"]}]}
    prefix = processor.apply_chat_template([user], tokenize=True, add_generation_prompt=True,
                                          truncation=False, return_dict=True, return_tensors="pt")
    full = processor.apply_chat_template([user, assistant], tokenize=True, add_generation_prompt=False,
                                        truncation=False, return_dict=True, return_tensors="pt")
    prefix_ids, ids = input_ids(prefix), input_ids(full)
    if not prefix_ids or ids[:len(prefix_ids)] != prefix_ids or len(ids) <= len(prefix_ids):
        raise ValueError("Chat-template prompt/target boundary mismatch; no truncation permitted")
    target_ids = ids[len(prefix_ids):]
    decoded = processor.tokenizer.decode(target_ids, skip_special_tokens=True)
    if " ".join(decoded.split()) != " ".join(row["completion"].split()):
        raise ValueError("Decoded completion tokens do not match the complete target")
    audit = {"id": row["id"], "paper_key": row["paper_key"], "split": row["split"],
             "prompt_tokens": len(prefix_ids), "target_tokens": len(target_ids), "full_tokens": len(ids),
             "prompt_sha256": digest(row["prompt"]),
             "prefix_token_ids_sha256": digest(json.dumps(prefix_ids)),
             "target_text_sha256": digest(row["completion"]),
             "boundary_matches": True, "decoded_target_matches": True, "truncated": False}
    return audit


def summarize_audit(records):
    if len(records) != 313 or dict(collections.Counter(r["split"] for r in records)) != COUNTS:
        raise ValueError("Token audit must cover every corpus row")
    maximum = max(r["full_tokens"] for r in records)
    recommended = next((n for n in (1024, 1536, 2048, 3072, 4096) if n >= maximum), None)
    validation = [r for r in records if r["split"] == "validation"]
    generation_budget = max(512, ((max(r["target_tokens"] for r in validation) + 64 + 127) // 128) * 128)
    return {"model_id": MODEL_ID, "dataset_sha256": DATASET_SHA256, "rows_audited": 313,
            "split_counts": COUNTS, "test_rows_length_audited": 45,
            "test_rows_generated": 0, "test_rows_scored": 0,
            "longest_full_sequence": maximum, "recommended_max_length": recommended,
            "recommendation_scope": "Fits all 313 labeled sequences; this is not a GPU memory-fit result",
            "baseline_max_new_tokens": generation_budget,
            "fits_context_budget": recommended is not None and generation_budget <= 1024 and
                all(r["prompt_tokens"] + generation_budget <= CONTEXT_BUDGET for r in validation),
            "limits": {str(n): {s: sum(r["full_tokens"] > n for r in records if r["split"] == s)
                                for s in COUNTS} for n in (1024, 1536, 2048, 3072, 4096)},
            "longest_train_ids": [r["id"] for r in sorted(
                (r for r in records if r["split"] == "train"), key=lambda r: (-r["full_tokens"], r["id"]))[:3]],
            "records": records}


def validate_result(result):
    if not result["output"].strip() or result["hit_output_limit"] or result["output_tokens"] <= 0:
        raise ValueError("Empty or cut-off generation; baseline remains incomplete")


def validate_baseline(report, validation, require_complete=True):
    if (report["model_id"] != MODEL_ID or report["dataset_sha256"] != DATASET_SHA256
            or report["prompt_format"] != FORMAT or report["instruction"] != INSTRUCTION
            or report["training_called"] is not False or report["test_rows_generated"] != 0
            or report["test_rows_scored"] != 0 or len(report["results"]) != 21
            or (require_complete and report.get("complete") is not True)
            or report.get("quantization") != "NF4 double quantization, FP32 compute"
            or report.get("demonstrations") != []
            or report.get("generation", {}).get("do_sample") is not False
            or not 0 < report.get("generation", {}).get("max_new_tokens", 0) <= 1024
            or not report.get("model_revision")
            or [r["id"] for r in report["results"]] != [r["id"] for r in validation]):
        raise ValueError("Baseline must contain exactly the frozen validation rows, in order")
    for result, row in zip(report["results"], validation):
        validate_result(result)
        if (result["prompt_sha256"] != digest(row["prompt"]) or result["source"] != row["plain_english"]
                or result["reference"] != row["completion"]):
            raise ValueError("Baseline output provenance does not match validation input")


def save_outputs(output_dir, report, validation):
    write_json(output_dir / "baseline.json", report)
    (output_dir / "baseline_generations.jsonl").write_text(
        "".join(json.dumps(r, ensure_ascii=False) + "\n" for r in report["results"]), encoding="utf-8")
    results = {r["id"]: r for r in report["results"]}
    lines = ["# Corpus-v1 base-model validation review", "",
             "Build the claim inventory from the plain-English source first. Sentence splits below are",
             "reading aids, not atomic claim labels. Judge preservation separately from style.",
             "References are for review only; the model received only the stored source prompt.", ""]
    forms = []
    for row in validation:
        generated = results.get(row["id"], {})
        sentences = re.split(r"(?<=[.!?])\s+(?=[A-Z])", row["plain_english"])
        lines.extend([f"## {row['id']}", "", "### Plain-English source", "", row["plain_english"], "",
                      "### Source reading aids", ""])
        lines.extend(f"- {s}" for s in sentences)
        lines.extend(["", "- Claim inventory: ", "", "### Base output", "",
                      generated.get("output", "Pending generation"), "", "### Original reference", "",
                      row["completion"], "", "### Claim preservation", "",
                      "- [ ] All substantive claims retained", "- [ ] No unsupported claims added",
                      "- [ ] Actors, causality, numbers, conditions and qualifiers preserved",
                      "- Added / lost / changed claims: ", "- Claim preservation score (0–4): ", "",
                      "### Style (score separately; 0–2 each)", "",
                      "- Precise economic terminology: ", "- Explicit analytical/conditional structure: ",
                      "- Concise exposition resembling the reference: ", "- Exact input copying: ",
                      "- Style evidence and limitations: ", ""])
        forms.append({"id": row["id"], "claim_inventory": [], "added_claims": [], "lost_claims": [],
                      "changed_claims": [], "claim_preservation_score": None,
                      "style": {"economic_terminology": None, "analytical_structure": None, "concision": None},
                      "review_status": "pending_manual_review"})
    (output_dir / "claim_style_review.md").write_text("\n".join(lines), encoding="utf-8")
    # Created once, so incremental saves never overwrite a review entered during a run.
    form_path = output_dir / "review_scores.jsonl"
    if not form_path.exists():
        form_path.write_text("".join(json.dumps(r) + "\n" for r in forms), encoding="utf-8")


def run_training(rows, groups, output_dir, audit_only=False):
    """Inference only. The name follows the project's GPU-import entry-point convention."""
    if output_dir.exists() and any(output_dir.iterdir()):
        raise ValueError("Output directory is nonempty; use a fresh Kaggle session or a new output path")
    output_dir.mkdir(parents=True, exist_ok=True)
    status = {"complete": False, "phase": "runtime_start", "training_called": False,
              "test_rows_generated": 0, "test_rows_scored": 0, "dataset_sha256": DATASET_SHA256,
              "implementation": IMPLEMENTATION}
    report = None
    records = []

    def phase(name, **extra):
        status.update(phase=name, **extra)
        write_json(output_dir / "run_status.json", status)
        print(f"Phase: {name}", flush=True)

    def load_with_heartbeat(loader, description):
        stop = threading.Event()
        started = time.monotonic()
        def heartbeat():
            while not stop.wait(30):
                print(f"{description}: still working ({int(time.monotonic() - started)} seconds)", flush=True)
        thread = threading.Thread(target=heartbeat, daemon=True)
        thread.start()
        try:
            return loader()
        finally:
            stop.set()
            thread.join(timeout=1)

    try:
        phase("loading_runtime")
        os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
        os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
        os.environ["TOKENIZERS_PARALLELISM"] = "false"
        os.environ["CUDA_VISIBLE_DEVICES"] = os.environ.get("CUDA_VISIBLE_DEVICES", "0").split(",")[0] or "0"
        import torch
        from transformers import (AutoConfig, AutoProcessor, BitsAndBytesConfig,
                                  Gemma3ForConditionalGeneration, LogitsProcessor, LogitsProcessorList)
        from huggingface_hub import HfApi
        if not audit_only and (not torch.cuda.is_available() or torch.cuda.device_count() != 1):
            raise RuntimeError("Select Kaggle GPU T4 x2; this notebook exposes and uses only its first GPU")
        phase("checking_secret")
        token = load_hf_token()
        phase("resolving_model_revision")
        config, revision = load_pinned_config(HfApi(token=token), AutoConfig, token, progress=phase)
        phase("loading_processor", model_id=MODEL_ID, model_revision=revision)
        processor = load_with_heartbeat(lambda: AutoProcessor.from_pretrained(
            MODEL_ID, revision=revision, token=token), "Processor download/load")
        tokenizer = processor.tokenizer
        phase("auditing_all_token_lengths")
        for index, row in enumerate(rows, 1):
            status["current_id"] = row["id"]
            records.append(prepare_pair(processor, row))
            if index % 50 == 0 or index == len(rows):
                print(f"Audited token lengths: {index}/{len(rows)}", flush=True)
        audit = summarize_audit(records)
        template = getattr(processor, "chat_template", None) or tokenizer.chat_template
        audit.update(model_revision=revision, chat_template_sha256=digest(json.dumps(template, sort_keys=True)),
                     tokenizer_class=type(tokenizer).__name__,
                     versions={name: importlib.metadata.version(name) for name in
                               ("torch", "transformers", "tokenizers", "accelerate", "huggingface-hub")},
                     implementation=IMPLEMENTATION)
        write_json(output_dir / "token_audit.json", audit)
        if not audit["fits_context_budget"]:
            raise ValueError("Sequence or generation budget exceeds preflight limits; token audit saved")
        print(f"No truncation: recommended max_length={audit['recommended_max_length']}; "
              f"longest sequence={audit['longest_full_sequence']}", flush=True)
        if audit_only:
            status["complete"] = True
            phase("token_audit_complete", model_loaded=False)
            return
        phase("initializing_cuda")
        initialize_cuda(torch)
        phase("loading_4b_model")
        model = load_with_heartbeat(lambda: Gemma3ForConditionalGeneration.from_pretrained(
            MODEL_ID, revision=revision, token=token, config=config,
            dtype=torch.float32, device_map={"": 0}, attn_implementation="sdpa",
            quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float32)), "4B model download/load")
        phase("checking_model_placement")
        placement = validate_model_placement(model)
        write_json(output_dir / "model_placement.json", placement)
        model.eval()
        model.requires_grad_(False)
        budget = audit["baseline_max_new_tokens"]
        by_id = {r["id"]: r for r in records}
        report = {"complete": False, "model_id": MODEL_ID, "model_revision": revision,
                  "implementation": IMPLEMENTATION,
                  "dataset_sha256": DATASET_SHA256, "prompt_format": FORMAT, "instruction": INSTRUCTION,
                  "training_called": False, "test_rows_length_audited": 45, "test_rows_generated": 0,
                  "test_rows_scored": 0, "validation_rows": 21, "versions": audit["versions"],
                  "chat_template_sha256": audit["chat_template_sha256"],
                  "quantization": "NF4 double quantization, FP32 compute", "seed": 42,
                  "demonstrations": [], "finite_generation_scores_checked": True,
                  "model_placement": placement,
                  "generation": {"do_sample": False, "max_new_tokens": budget, "use_cache": True},
                  "recommended_max_length": audit["recommended_max_length"], "results": []}
        save_outputs(output_dir, report, groups["validation"])

        class FiniteScores(LogitsProcessor):
            def __call__(self, input_ids, scores):
                if bool(torch.isnan(scores).any() | torch.isposinf(scores).any()
                        | ~torch.isfinite(scores).any(dim=-1).all()):
                    raise RuntimeError("Invalid generation scores; baseline remains incomplete")
                return scores

        for index, row in enumerate(groups["validation"], 1):
            phase("validation_generation", saved_generations=index - 1, current_id=row["id"])
            inputs = processor.apply_chat_template([user_message(row)], tokenize=True, add_generation_prompt=True,
                truncation=False, return_dict=True, return_tensors="pt")
            ids = input_ids(inputs)
            if digest(json.dumps(ids)) != by_id[row["id"]]["prefix_token_ids_sha256"]:
                raise ValueError("Generation prompt differs from audited prompt")
            if len(ids) + budget > CONTEXT_BUDGET:
                raise ValueError("Generation context over budget; no truncation permitted")
            inputs = {k: v.to(model.get_input_embeddings().weight.device) for k, v in inputs.items()}
            started = time.monotonic()
            with torch.inference_mode():
                tokens = load_with_heartbeat(lambda: model.generate(**inputs, do_sample=False,
                    max_new_tokens=budget, use_cache=True, pad_token_id=tokenizer.pad_token_id,
                    remove_invalid_values=False, logits_processor=LogitsProcessorList([FiniteScores()])),
                    f"Validation generation {index}/21")
            generated = tokens[0, len(ids):]
            output = tokenizer.decode(generated, skip_special_tokens=True).strip()
            eos = model.generation_config.eos_token_id
            eos = eos if isinstance(eos, list) else [eos]
            result = {"id": row["id"], "paper_key": row["paper_key"], "split": "validation",
                      "source": row["plain_english"], "reference": row["completion"], "output": output,
                      "prompt_sha256": digest(row["prompt"]), "input_tokens": len(ids),
                      "output_tokens": int(generated.numel()), "first_generated_token_ids": generated[:16].tolist(),
                      "hit_output_limit": generated.numel() >= budget and
                          (not generated.numel() or int(generated[-1]) not in eos),
                      "copied_source": " ".join(output.split()) == " ".join(row["plain_english"].split()),
                      "generation_seconds": time.monotonic() - started, "claim_review_status": "pending_manual_review"}
            report["results"].append(result)
            save_outputs(output_dir, report, groups["validation"])
            validate_result(result)
            print(f"Saved valid validation output {index}/21 ({result['output_tokens']} tokens)", flush=True)
        validate_baseline(report, groups["validation"], require_complete=False)
        report["memory"] = {"gpu": torch.cuda.get_device_name(0),
                            "peak_allocated_bytes": torch.cuda.max_memory_allocated(0),
                            "peak_reserved_bytes": torch.cuda.max_memory_reserved(0)}
        report["complete"] = True
        save_outputs(output_dir, report, groups["validation"])
        artifacts = {name: hashlib.sha256((output_dir / name).read_bytes()).hexdigest()
                     for name in ("baseline.json", "baseline_generations.jsonl", "token_audit.json")}
        write_json(output_dir / "baseline_manifest.json", {"model_id": MODEL_ID, "model_revision": revision,
                   "dataset_sha256": DATASET_SHA256, "files": artifacts})
        status["complete"] = True
        phase("baseline_complete", saved_generations=21)
    except Exception as exc:
        failed_phase = status["phase"]
        status.update(complete=False, phase="failed", failed_phase=failed_phase, error_class=type(exc).__name__)
        status.update(safe_failure_details(exc))
        if isinstance(exc, PreflightError):
            status["error_code"] = exc.code
        if failed_phase == "auditing_all_token_lengths":
            write_json(output_dir / "token_audit_partial.json", {"complete": False,
                       "rows_audited": len(records), "failed_id": status.get("current_id"),
                       "dataset_sha256": DATASET_SHA256, "records": records})
        hints = {
            "checking_secret": "Enable the HF_TOKEN Kaggle Secret for this notebook.",
            "resolving_model_revision": "Check internet and Hugging Face model access; this is a Hub revision lookup.",
            "loading_model_config": "Confirm the Gemma license is accepted, model access is permitted, and internet is enabled.",
            "initializing_cuda": "CUDA initialization or memory-counter setup failed before weight loading; inspect the safe stack locations.",
            "loading_4b_model": "Weight loading failed; inspect the safe stack locations and error code. No raw remote exception text is saved.",
        }
        if failed_phase in hints:
            status["hint"] = hints[failed_phase]
        write_json(output_dir / "run_status.json", status)
        # Remote exception messages/URLs can contain credentials. Retain phase/class only.
        hint = " " + status["hint"] if "hint" in status else ""
        raise RuntimeError(f"Preflight stopped in {failed_phase} ({type(exc).__name__}); "
                           "see run_status.json and any saved partial artifacts. No training was called."
                           + hint) from None


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--dataset", required=True)
    parser.add_argument("--output-dir", default="/kaggle/working/corpus_v1_baseline")
    parser.add_argument("--check", action="store_true", help="CPU-only schema/hash/split checks; no ML imports")
    parser.add_argument("--audit-only", action="store_true", help="Download processor/tokenizer only; skip model weights")
    args = parser.parse_args(argv)
    rows, groups = load_corpus(args.dataset)
    print("Validated frozen corpus: 247 train / 21 validation / 45 test")
    if args.check:
        print("Dry-run OK. No credentials accessed, model downloaded, generation or training performed.")
        return
    run_training(rows, groups, Path(args.output_dir), args.audit_only)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/corpus_smoke.py
"""Three-row, five-update 4B QLoRA pipeline test against corpus baseline v4."""

import argparse
import hashlib
import importlib.metadata
import json
import math
import os
import re
import threading
from pathlib import Path

import corpus_baseline as baseline

IMPLEMENTATION = "corpus_smoke_v1"
BASELINE_MANIFEST_SHA256 = "929ab47278132ce47176d81b0fbff38cf2d8ceb7c1e40c90b790d0fe525492e9"
MODEL_REVISION = "093f9f388b31de276ce2de164bdc2081324b9767"
MAX_LENGTH = 1024
STEPS = 5
ACCUMULATION = 2
TARGET_PATTERN = r".*language_model\.layers\.\d+\.self_attn\.(q_proj|v_proj)$"


def load_inputs(dataset, baseline_dir):
    rows, groups = baseline.load_corpus(dataset)
    directory = Path(baseline_dir)
    manifest_bytes = (directory / "baseline_manifest.json").read_bytes()
    if hashlib.sha256(manifest_bytes).hexdigest() != BASELINE_MANIFEST_SHA256:
        raise ValueError("Attach the frozen successful corpus-v1 baseline version 4 output")
    manifest = json.loads(manifest_bytes)
    for name in ("baseline.json", "baseline_generations.jsonl", "token_audit.json"):
        if hashlib.sha256((directory / name).read_bytes()).hexdigest() != manifest["files"][name]:
            raise ValueError("Baseline artifact hash mismatch")
    report = json.loads((directory / "baseline.json").read_text())
    baseline.validate_baseline(report, groups["validation"])
    audit = json.loads((directory / "token_audit.json").read_text())
    records = audit["records"]
    if (report["model_revision"] != MODEL_REVISION or audit["model_revision"] != MODEL_REVISION
            or audit["dataset_sha256"] != baseline.DATASET_SHA256
            or audit["recommended_max_length"] != MAX_LENGTH
            or [r["id"] for r in records] != [r["id"] for r in rows]):
        raise ValueError("Baseline revision, audit or row provenance changed")
    train_ids, validation_ids = choose_examples(groups, records)
    return groups, audit, train_ids, validation_ids


def choose_examples(groups, records):
    by_id = {r["id"]: r for r in records}
    def ranked(split):
        return sorted(groups[split], key=lambda row: (-by_id[row["id"]]["full_tokens"], row["id"]))
    train_ids = [r["id"] for r in ranked("train")[:3]]
    validation_ids = [r["id"] for r in ranked("validation")[:2]]
    if (len(set(train_ids)) != 3 or len(set(validation_ids)) != 2
            or set(train_ids) & set(validation_ids)
            or any(by_id[key]["split"] != split or by_id[key]["full_tokens"] > MAX_LENGTH
                   for split, ids in (("train", train_ids), ("validation", validation_ids)) for key in ids)):
        raise ValueError("Smoke needs three unique training and two disjoint validation rows within 1024 tokens")
    return train_ids, validation_ids


def encode_pair(processor, row, expected):
    audit = baseline.prepare_pair(processor, row)
    if audit != expected or audit["full_tokens"] > MAX_LENGTH:
        raise ValueError("Runtime tokenization differs from the frozen baseline audit")
    messages = [baseline.user_message(row), {"role": "assistant", "content": [
        {"type": "text", "text": row["completion"]}]}]
    batch = processor.apply_chat_template(messages, tokenize=True, add_generation_prompt=False,
        truncation=False, return_dict=True, return_tensors="pt")
    ids = baseline.input_ids(batch)
    # Revalidate the actual training batch, not only a separately rendered audit.
    prefix = audit["prompt_tokens"]
    if (len(ids) != audit["full_tokens"]
            or baseline.digest(json.dumps(ids[:prefix])) != audit["prefix_token_ids_sha256"]
            or " ".join(processor.tokenizer.decode(ids[prefix:], skip_special_tokens=True).split())
               != " ".join(row["completion"].split())):
        raise ValueError("Actual training batch changed its prefix or supervised target")
    labels = [-100] * prefix + ids[prefix:]
    prepared = {name: baseline.input_ids({"input_ids": value}) for name, value in batch.items()
                if name in ("input_ids", "attention_mask", "token_type_ids")}
    prepared["labels"] = labels
    audit = dict(audit, masked_prompt_tokens=prefix, supervised_tokens=len(ids)-prefix,
                 shifted_supervised_tokens=sum(value != -100 for value in labels[1:]))
    return prepared, audit


def validate_completion(status, evaluation):
    if (status.get("complete") is not True or status.get("optimizer_steps_completed") != STEPS
            or status.get("test_rows_used") != 0 or status.get("adapter_weights_changed") is not True
            or status.get("adapter_reload_verified") is not True
            or len(set(status.get("trained_ids", []))) != 3
            or len(set(status.get("validation_ids", []))) != 2
            or set(status["trained_ids"]) & set(status["validation_ids"])
            or len(status.get("updates", [])) != STEPS):
        raise ValueError("Incomplete smoke test; five updates and adapter reload are required")
    for index, update in enumerate(status["updates"], 1):
        expected_ids = [status["trained_ids"][((index-1)*ACCUMULATION+i) % 3] for i in range(ACCUMULATION)]
        if (update["step"] != index or len(update["microbatch_ids"]) != ACCUMULATION
                or update["microbatch_ids"] != expected_ids
                or any(key not in status["trained_ids"] for key in update["microbatch_ids"])
                or not math.isfinite(update["loss"]) or not math.isfinite(update["gradient_norm"])
                or update["gradient_norm"] <= 0):
            raise ValueError("Invalid update, loss or gradients")
    for key in ("original_validation_loss", "fine_tuned_validation_loss", "reloaded_validation_loss"):
        if not math.isfinite(evaluation[key]):
            raise ValueError("Non-finite held-out loss")
    if (evaluation["validation_ids"] != status["validation_ids"] or evaluation["test_rows_used"] != 0
            or evaluation.get("optimizer_steps") != STEPS or evaluation.get("train_rows") != 3
            or evaluation.get("adapter_reload_verified") is not True
            or not math.isclose(evaluation["fine_tuned_validation_loss"], evaluation["reloaded_validation_loss"],
                                rel_tol=1e-5, abs_tol=1e-5)):
        raise ValueError("Reloaded evaluation must match the trained adapter on the same held-out rows")


def run_training(groups, frozen_audit, train_ids, validation_ids, output_dir):
    if output_dir.exists() and any(output_dir.iterdir()):
        raise ValueError("Use a fresh output directory; saved smoke results must not be overwritten")
    output_dir.mkdir(parents=True, exist_ok=True)
    status = {"implementation": IMPLEMENTATION, "complete": False, "phase": "starting",
              "purpose": "pipeline and memory smoke test; not a learning-quality experiment",
              "model_id": baseline.MODEL_ID, "model_revision": MODEL_REVISION,
              "dataset_sha256": baseline.DATASET_SHA256, "baseline_manifest_sha256": BASELINE_MANIFEST_SHA256,
              "trained_ids": train_ids, "validation_ids": validation_ids, "test_rows_used": 0,
              "max_length": MAX_LENGTH, "optimizer_steps_planned": STEPS, "optimizer_steps_completed": 0,
              "gradient_accumulation_steps": ACCUMULATION, "microbatch_size": 1,
              "learning_rate": 5e-5, "lora_rank": 8, "lora_alpha": 16, "lora_dropout": 0.05,
              "target_modules_regex": TARGET_PATTERN, "seed": 42,
              "quantization": "NF4 double quantization, FP32 compute", "updates": [],
              "training_called": False, "training_gpus_used": 1, "second_t4_used": False}
    status_path = output_dir / "smoke_status.json"

    def phase(name, **values):
        status.update(phase=name, **values)
        baseline.write_json(status_path, status)
        print("Phase: " + name, flush=True)

    def heartbeat(operation, label):
        stop = threading.Event()
        def announce():
            while not stop.wait(30):
                print(label + ": still working", flush=True)
        thread = threading.Thread(target=announce, daemon=True)
        thread.start()
        try:
            return operation()
        finally:
            stop.set()
            thread.join(timeout=1)

    try:
        phase("loading_runtime")
        os.environ["CUDA_VISIBLE_DEVICES"] = "0"
        os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
        os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
        os.environ["TOKENIZERS_PARALLELISM"] = "false"
        import torch
        from transformers import AutoConfig, AutoProcessor, BitsAndBytesConfig, Gemma3ForConditionalGeneration
        from peft import LoraConfig, PeftModel, get_peft_model, get_peft_model_state_dict, prepare_model_for_kbit_training

        status["versions"] = {name: importlib.metadata.version(name) for name in
                              ("torch", "transformers", "peft", "bitsandbytes", "accelerate", "huggingface-hub")}
        if not torch.cuda.is_available() or torch.cuda.device_count() != 1:
            raise baseline.PreflightError("one_visible_cuda_gpu_required")
        phase("checking_secret")
        token = baseline.load_hf_token()
        phase("loading_pinned_config")
        config = AutoConfig.from_pretrained(baseline.MODEL_ID, revision=MODEL_REVISION, token=token)
        phase("loading_processor")
        processor = heartbeat(lambda: AutoProcessor.from_pretrained(
            baseline.MODEL_ID, revision=MODEL_REVISION, token=token), "Processor load")
        phase("checking_supervised_tokens")
        rows = {row["id"]: row for split in ("train", "validation") for row in groups[split]}
        expected = {row["id"]: row for row in frozen_audit["records"]}
        prepared, audits = {}, []
        for key in train_ids + validation_ids:
            prepared[key], audit = encode_pair(processor, rows[key], expected[key])
            audits.append(audit)
        baseline.write_json(output_dir / "prepared_label_audit.json", {"records": audits, "truncation": False})
        phase("initializing_cuda")
        baseline.initialize_cuda(torch)
        phase("loading_4b_model")
        model = heartbeat(lambda: Gemma3ForConditionalGeneration.from_pretrained(
            baseline.MODEL_ID, revision=MODEL_REVISION, token=token, config=config,
            dtype=torch.float32, device_map={"": 0}, attn_implementation="sdpa",
            quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float32)), "4B model load")
        phase("checking_model_placement")
        status["base_placement"] = baseline.validate_model_placement(model)
        phase("preparing_qlora")
        model.config.use_cache = False
        model.config.text_config.use_cache = False
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True,
            gradient_checkpointing_kwargs={"use_reentrant": False})
        matched = [name for name, _ in model.named_modules() if re.fullmatch(TARGET_PATTERN, name)]
        if len(matched) != 2 * config.text_config.num_hidden_layers:
            raise baseline.PreflightError("language_qv_adapter_module_count_changed")
        model = get_peft_model(model, LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05,
            target_modules=TARGET_PATTERN, bias="none", task_type="CAUSAL_LM", revision=MODEL_REVISION))
        parameters = {name: value for name, value in model.named_parameters() if value.requires_grad}
        if not parameters or any("lora_" not in name or "language_model" not in name for name in parameters):
            raise baseline.PreflightError("only_language_lora_parameters_may_train")
        status.update(trainable_parameters=sum(p.numel() for p in parameters.values()),
                      adapted_language_modules=len(matched), adapted_placement=baseline.validate_model_placement(model))

        def adapter_hashes():
            return {name: hashlib.sha256(value.detach().float().cpu().contiguous().numpy().tobytes()).hexdigest()
                    for name, value in get_peft_model_state_dict(model, save_embedding_layers=False).items()}

        initial_hashes = adapter_hashes()

        def batch(key):
            return {name: torch.tensor([value], dtype=torch.long, device="cuda:0")
                    for name, value in prepared[key].items()}

        target_counts = {a["id"]: a["shifted_supervised_tokens"] for a in audits}

        def evaluate():
            model.eval()
            values = []
            with torch.inference_mode():
                for key in validation_ids:
                    loss = float(model(**batch(key), use_cache=False).loss.item())
                    if not math.isfinite(loss):
                        raise baseline.PreflightError("nonfinite_validation_loss")
                    values.append({"id": key, "loss": loss, "supervised_tokens": target_counts[key]})
            return {"loss": sum(v["loss"]*v["supervised_tokens"] for v in values)
                            / sum(v["supervised_tokens"] for v in values), "rows": values}

        phase("base_validation_loss")
        with model.disable_adapter():
            original = heartbeat(evaluate, "Base validation loss")
        baseline.write_json(output_dir / "base_validation_loss.json", original)
        optimizer = torch.optim.AdamW(parameters.values(), lr=5e-5, weight_decay=0.0)
        phase("training", training_called=True)
        for step in range(STEPS):
            model.train()
            optimizer.zero_grad(set_to_none=True)
            used, losses = [], []
            for micro in range(ACCUMULATION):
                key = train_ids[(step*ACCUMULATION+micro) % len(train_ids)]
                def forward_backward():
                    loss = model(**batch(key), use_cache=False).loss
                    if not bool(torch.isfinite(loss).all()):
                        raise baseline.PreflightError("nonfinite_training_loss")
                    value = float(loss.detach().item())
                    (loss / ACCUMULATION).backward()
                    return value
                losses.append(heartbeat(forward_backward, f"Step {step+1}/{STEPS}, microbatch {micro+1}/2"))
                used.append(key)
            gradients = [p.grad for p in parameters.values() if p.grad is not None]
            if not gradients or any(not bool(torch.isfinite(g).all()) for g in gradients):
                raise baseline.PreflightError("missing_or_nonfinite_adapter_gradients")
            norm = float(torch.nn.utils.clip_grad_norm_(list(parameters.values()), 1.0, error_if_nonfinite=True))
            if not math.isfinite(norm) or norm <= 0:
                raise baseline.PreflightError("adapter_gradient_norm_not_positive")
            optimizer.step()
            if any(not bool(torch.isfinite(p).all()) for p in parameters.values()):
                raise baseline.PreflightError("nonfinite_adapter_weights")
            status["updates"].append({"step": step+1, "microbatch_ids": used,
                                      "loss": sum(losses)/len(losses), "gradient_norm": norm})
            status["optimizer_steps_completed"] = step+1
            phase("training")
            print(f"Completed optimizer update {step+1}/{STEPS}; loss={sum(losses)/len(losses):.6f}", flush=True)
        optimizer.zero_grad(set_to_none=True)
        phase("adapted_validation_loss")
        adapted = heartbeat(evaluate, "Adapted validation loss")
        trained_hashes = adapter_hashes()
        if trained_hashes == initial_hashes:
            raise baseline.PreflightError("adapter_weights_unchanged")
        status["adapter_weights_changed"] = True
        phase("saving_adapter")
        adapter_dir = output_dir / "adapter"
        # Vocabulary/embeddings are unchanged and frozen. Avoid the optional
        # automatic Hub config lookup used to detect resized embeddings.
        model.save_pretrained(adapter_dir, safe_serialization=True, save_embedding_layers=False)
        processor.save_pretrained(adapter_dir)
        baseline.write_json(output_dir / "adapter_tensor_hashes.json", trained_hashes)
        # Remove trained adapter modules, then load the files onto the same frozen
        # quantized base. Do not merge or allocate a second 4B base on the T4.
        phase("reloading_adapter")
        del optimizer, parameters, gradients
        bare_model = model.unload()
        del model
        if any("lora_" in name for name, _ in bare_model.named_parameters()):
            raise baseline.PreflightError("adapter_unload_incomplete")
        model = PeftModel.from_pretrained(bare_model, adapter_dir, is_trainable=False)
        if adapter_hashes() != trained_hashes:
            raise baseline.PreflightError("reloaded_adapter_tensor_hash_mismatch")
        status["reloaded_placement"] = baseline.validate_model_placement(model)
        phase("reloaded_validation_loss")
        reloaded = heartbeat(evaluate, "Reloaded validation loss")
        for before, after in zip(adapted["rows"], reloaded["rows"]):
            if before["id"] != after["id"] or not math.isclose(before["loss"], after["loss"], rel_tol=1e-5, abs_tol=1e-5):
                raise baseline.PreflightError("reloaded_validation_loss_mismatch")
        status.update(adapter_reload_verified=True, memory={"gpu": torch.cuda.get_device_name(0),
            "peak_allocated_bytes": torch.cuda.max_memory_allocated(0),
            "peak_reserved_bytes": torch.cuda.max_memory_reserved(0)})
        evaluation = {"purpose": status["purpose"], "model_id": baseline.MODEL_ID,
            "model_revision": MODEL_REVISION, "dataset_sha256": baseline.DATASET_SHA256,
            "train_rows": 3, "optimizer_steps": STEPS, "validation_ids": validation_ids,
            "test_rows_used": 0, "loss_aggregation": "shifted supervised-token weighted mean across two rows",
            "original_validation_loss": original["loss"], "fine_tuned_validation_loss": adapted["loss"],
            "reloaded_validation_loss": reloaded["loss"], "original": original, "fine_tuned": adapted,
            "reloaded": reloaded, "adapter_reload_verified": True,
            "style_improvement_claimed": False, "generations": []}
        status["complete"] = True
        validate_completion(status, evaluation)
        baseline.write_json(output_dir / "evaluation.json", evaluation)
        baseline.write_json(output_dir / "training_log.json", status["updates"])
        phase("complete")
        print("Smoke complete: 3 training rows, 5 updates, adapter reloaded; evaluation.json saved", flush=True)
    except Exception as exc:
        failed_phase = status["phase"]
        status.update(complete=False, phase="failed", failed_phase=failed_phase, error_class=type(exc).__name__)
        status.update(baseline.safe_failure_details(exc))
        tb = exc.__traceback__
        while tb is not None:
            if tb.tb_frame.f_code.co_filename == __file__:
                status["stack_locations"].append({"component": "corpus_smoke", "line": tb.tb_lineno})
            tb = tb.tb_next
        if isinstance(exc, baseline.PreflightError):
            status["error_code"] = exc.code
        baseline.write_json(status_path, status)
        raise RuntimeError(f"Smoke stopped in {failed_phase}; inspect smoke_status.json. "
                           "Remote exception messages are suppressed.") from None


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--dataset", required=True)
    parser.add_argument("--baseline-dir", required=True)
    parser.add_argument("--output-dir", default="/kaggle/working/corpus_v1_smoke")
    parser.add_argument("--check", action="store_true")
    args = parser.parse_args(argv)
    groups, audit, train_ids, validation_ids = load_inputs(args.dataset, args.baseline_dir)
    print("Validated frozen corpus and successful baseline v4; smoke IDs:", train_ids, validation_ids)
    if args.check:
        print("Dry-run OK. No Secret accessed, model downloaded or training called.")
        return
    run_training(groups, audit, train_ids, validation_ids, Path(args.output_dir))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/corpus_pilot.py
"""One-epoch 247-pair 4B QLoRA pilot with frozen 21-row base comparison."""

import argparse
import hashlib
import importlib.metadata
import json
import math
import os
import random
import re
import threading
from pathlib import Path

import corpus_baseline as baseline
import corpus_smoke as smoke

IMPLEMENTATION = "corpus_pilot_one_epoch_v1"
SMOKE_HASHES = {
    "smoke_status.json": "c79cddd061750a5430cdcf65bac37cd1bab64eed19168d83a327fbdac214596b",
    "evaluation.json": "a8c1daa71f68dbdd38bd13ba621cdf66a36ffcc5bf9546e7ddc3750f566e38b2",
}


def load_inputs(dataset, baseline_dir, smoke_dir):
    groups, audit, train_ids, validation_ids = smoke.load_inputs(dataset, baseline_dir)
    smoke_dir = Path(smoke_dir)
    for name, expected in SMOKE_HASHES.items():
        if hashlib.sha256((smoke_dir / name).read_bytes()).hexdigest() != expected:
            raise ValueError("Attach the successful corpus-v1 smoke version 1 output")
    status = json.loads((smoke_dir / "smoke_status.json").read_text())
    evaluation = json.loads((smoke_dir / "evaluation.json").read_text())
    smoke.validate_completion(status, evaluation)
    if (status["dataset_sha256"] != baseline.DATASET_SHA256
            or status["model_revision"] != smoke.MODEL_REVISION
            or status["trained_ids"] != train_ids or status["validation_ids"] != validation_ids):
        raise ValueError("Smoke provenance does not match frozen corpus/baseline")
    original = json.loads((Path(baseline_dir) / "baseline.json").read_text())
    return groups, audit, original


def epoch_batches(train_ids):
    if len(train_ids) != 247 or len(set(train_ids)) != 247:
        raise ValueError("One epoch requires 247 unique training IDs")
    order = list(train_ids)
    random.Random(42).shuffle(order)
    return [order[i:i+smoke.ACCUMULATION] for i in range(0, len(order), smoke.ACCUMULATION)]


def validate_loss_report(report, validation):
    if [r["id"] for r in report["rows"]] != [r["id"] for r in validation]:
        raise ValueError("Loss report must cover exactly all 21 validation rows")
    if any(not math.isfinite(r["loss"]) or r["loss"] < 0 or r["supervised_tokens"] <= 0 for r in report["rows"]):
        raise ValueError("Invalid validation loss or supervised-token count")
    total = sum(r["supervised_tokens"] for r in report["rows"])
    expected = sum(r["loss"]*r["supervised_tokens"] for r in report["rows"])/total
    if not math.isfinite(report["loss"]) or not math.isclose(report["loss"], expected, rel_tol=1e-10):
        raise ValueError("Invalid token-weighted aggregate validation loss")


def validate_comparison(comparison, validation, original, require_complete=True):
    if (comparison.get("complete") is not True and require_complete
            or comparison["model_id"] != baseline.MODEL_ID
            or comparison["model_revision"] != smoke.MODEL_REVISION
            or comparison["dataset_sha256"] != baseline.DATASET_SHA256
            or comparison["test_rows_used"] != 0 or comparison["epochs"] != 1
            or comparison["generation"] != original["generation"]
            or len(comparison["results"]) != 21):
        raise ValueError("Comparison must cover the frozen 21-row validation set")
    for result, row, old in zip(comparison["results"], validation, original["results"]):
        baseline.validate_result(result)
        if (result["id"] != row["id"] or result["source"] != row["plain_english"]
                or result["reference"] != row["completion"] or result["baseline_output"] != old["output"]
                or result["prompt_sha256"] != baseline.digest(row["prompt"])):
            raise ValueError("Comparison provenance or prompt changed")


def validate_completion(status, evaluation, comparison, groups, original):
    expected = epoch_batches([row["id"] for row in groups["train"]])
    if (status.get("complete") is not True or status.get("epochs_completed") != 1
            or status.get("epochs_planned") != 1 or status.get("training_called") is not True
            or status.get("optimizer_steps_completed") != len(expected)
            or status.get("training_examples_seen") != 247
            or status.get("adapter_weights_changed") is not True
            or status.get("adapter_reload_verified") is not True or status.get("test_rows_used") != 0
            or status.get("train_order") != [key for batch in expected for key in batch]
            or len(status.get("updates", [])) != len(expected)):
        raise ValueError("Pilot must complete exactly one epoch with verified adapter reload")
    for step, (update, ids) in enumerate(zip(status["updates"], expected), 1):
        if (update["step"] != step or update["microbatch_ids"] != ids
                or not math.isfinite(update["loss"]) or not math.isfinite(update["gradient_norm"])
                or update["gradient_norm"] <= 0):
            raise ValueError("Invalid pilot update, training coverage or gradients")
    for key in ("original", "fine_tuned", "reloaded"):
        validate_loss_report(evaluation[key], groups["validation"])
    for before, after in zip(evaluation["fine_tuned"]["rows"], evaluation["reloaded"]["rows"]):
        if before["supervised_tokens"] != after["supervised_tokens"] or not math.isclose(
                before["loss"], after["loss"], rel_tol=1e-5, abs_tol=1e-5):
            raise ValueError("Reloaded per-row losses do not match")
    for label, key in (("original_validation_loss", "original"),
                       ("fine_tuned_validation_loss", "fine_tuned"), ("reloaded_validation_loss", "reloaded")):
        if evaluation[label] != evaluation[key]["loss"]:
            raise ValueError("Validation summary differs from per-row loss report")
    if (evaluation.get("complete") is not True or evaluation["test_rows_used"] != 0 or evaluation["epochs"] != 1
            or evaluation["optimizer_steps"] != 124
            or evaluation["train_rows"] != 247 or evaluation["adapter_reload_verified"] is not True):
        raise ValueError("Evaluation scope or reload status changed")
    validate_comparison(comparison, groups["validation"], original)


def save_comparison(directory, comparison):
    baseline.write_json(directory / "comparison.json", comparison)
    (directory / "validation_generations.jsonl").write_text("".join(
        json.dumps(row, ensure_ascii=False)+"\n" for row in comparison["results"]), encoding="utf-8")


def save_review(directory, comparison):
    """Masked-order worksheet; mapping stays in a separate file, not independently blind."""
    rng = random.Random(20261019)
    lines = ["# One-epoch validation comparison", "",
             "Review claim preservation and style separately. Build a claim inventory from the source.",
             "A/B order varies by paragraph. Keep comparison_key.json closed until scores are saved.",
             "This masks labels only; it does not establish independent blinded evaluation.", ""]
    mapping, forms = {}, []
    for result in comparison["results"]:
        labels = ["base", "epoch_1"]
        rng.shuffle(labels)
        mapping[result["id"]] = dict(zip(("A", "B"), labels))
        outputs = {"base": result["baseline_output"], "epoch_1": result["output"]}
        lines += ["## "+result["id"], "", "### Source", "", result["source"], "",
                  "### Original reference", "", result["reference"], ""]
        for slot, label in zip(("A", "B"), labels):
            lines += ["### Candidate "+slot, "", outputs[label], "",
                      "- Claim inventory / additions / omissions / changed conditions, actors, numbers or causality:",
                      "- Claim preservation (0–4):", "- Economic precision (0–2):",
                      "- Analytical structure (0–2):", "- Concision resembling the reference (0–2):",
                      "- Style evidence; copying and verbosity observations:", ""]
            forms.append({"id": result["id"], "candidate": slot, "claim_inventory": [],
                "added_claims": [], "lost_claims": [], "changed_claims": [], "claim_preservation_score": None,
                "style": {"economic_precision": None, "analytical_structure": None, "concision": None},
                "review_status": "pending_review"})
    (directory / "claim_style_review.md").write_text("\n".join(lines), encoding="utf-8")
    baseline.write_json(directory / "comparison_key.json", mapping)
    path = directory / "review_scores.jsonl"
    if not path.exists():
        path.write_text("".join(json.dumps(form)+"\n" for form in forms), encoding="utf-8")


def run_training(groups, frozen_audit, original_baseline, output_dir):
    train_ids = [row["id"] for row in groups["train"]]
    validation_ids = [row["id"] for row in groups["validation"]]
    schedule = epoch_batches(train_ids)
    train_order = [key for ids in schedule for key in ids]
    if output_dir.exists() and any(output_dir.iterdir()):
        raise ValueError("Use a fresh output directory; saved pilot results must not be overwritten")
    output_dir.mkdir(parents=True, exist_ok=True)
    status = {"implementation": IMPLEMENTATION, "complete": False, "phase": "starting",
              "purpose": "bounded one-epoch pilot; no automatic continuation",
              "model_id": baseline.MODEL_ID, "model_revision": smoke.MODEL_REVISION,
              "dataset_sha256": baseline.DATASET_SHA256, "baseline_manifest_sha256": smoke.BASELINE_MANIFEST_SHA256,
              "trained_ids": train_ids, "validation_ids": validation_ids, "test_rows_used": 0,
              "max_length": smoke.MAX_LENGTH, "optimizer_steps_planned": len(schedule), "optimizer_steps_completed": 0,
              "epochs_planned": 1, "epochs_completed": 0, "training_examples_seen": 0, "train_order": train_order,
              "gradient_accumulation_steps": smoke.ACCUMULATION, "microbatch_size": 1,
              "learning_rate": 5e-5, "lora_rank": 8, "lora_alpha": 16, "lora_dropout": 0.05,
              "target_modules_regex": smoke.TARGET_PATTERN, "seed": 42,
              "quantization": "NF4 double quantization, FP32 compute", "updates": [],
              "training_called": False, "training_gpus_used": 1, "second_t4_used": False}
    status_path = output_dir / "pilot_status.json"

    def phase(name, **values):
        status.update(phase=name, **values)
        baseline.write_json(status_path, status)
        print("Phase: " + name, flush=True)

    def heartbeat(operation, label):
        stop = threading.Event()
        def announce():
            while not stop.wait(30):
                print(label + ": still working", flush=True)
        thread = threading.Thread(target=announce, daemon=True)
        thread.start()
        try:
            return operation()
        finally:
            stop.set()
            thread.join(timeout=1)

    try:
        phase("loading_runtime")
        os.environ["CUDA_VISIBLE_DEVICES"] = "0"
        os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
        os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
        os.environ["TOKENIZERS_PARALLELISM"] = "false"
        import torch
        from transformers import (AutoConfig, AutoProcessor, BitsAndBytesConfig, Gemma3ForConditionalGeneration,
                                  LogitsProcessor, LogitsProcessorList)
        from peft import LoraConfig, PeftModel, get_peft_model, get_peft_model_state_dict, prepare_model_for_kbit_training

        status["versions"] = {name: importlib.metadata.version(name) for name in
                              ("torch", "transformers", "peft", "bitsandbytes", "accelerate", "huggingface-hub")}
        if not torch.cuda.is_available() or torch.cuda.device_count() != 1:
            raise baseline.PreflightError("one_visible_cuda_gpu_required")
        phase("checking_secret")
        token = baseline.load_hf_token()
        phase("loading_pinned_config")
        config = AutoConfig.from_pretrained(baseline.MODEL_ID, revision=smoke.MODEL_REVISION, token=token)
        phase("loading_processor")
        processor = heartbeat(lambda: AutoProcessor.from_pretrained(
            baseline.MODEL_ID, revision=smoke.MODEL_REVISION, token=token), "Processor load")
        phase("checking_supervised_tokens")
        rows = {row["id"]: row for split in ("train", "validation") for row in groups[split]}
        expected = {row["id"]: row for row in frozen_audit["records"]}
        prepared, audits = {}, []
        for key in train_ids + validation_ids:
            prepared[key], audit = smoke.encode_pair(processor, rows[key], expected[key])
            audits.append(audit)
        baseline.write_json(output_dir / "prepared_label_audit.json", {"records": audits, "truncation": False})
        phase("initializing_cuda")
        baseline.initialize_cuda(torch)
        phase("loading_4b_model")
        model = heartbeat(lambda: Gemma3ForConditionalGeneration.from_pretrained(
            baseline.MODEL_ID, revision=smoke.MODEL_REVISION, token=token, config=config,
            dtype=torch.float32, device_map={"": 0}, attn_implementation="sdpa",
            quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float32)), "4B model load")
        phase("checking_model_placement")
        status["base_placement"] = baseline.validate_model_placement(model)
        phase("preparing_qlora")
        model.config.use_cache = False
        model.config.text_config.use_cache = False
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True,
            gradient_checkpointing_kwargs={"use_reentrant": False})
        matched = [name for name, _ in model.named_modules() if re.fullmatch(smoke.TARGET_PATTERN, name)]
        if len(matched) != 2 * config.text_config.num_hidden_layers:
            raise baseline.PreflightError("language_qv_adapter_module_count_changed")
        model = get_peft_model(model, LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05,
            target_modules=smoke.TARGET_PATTERN, bias="none", task_type="CAUSAL_LM", revision=smoke.MODEL_REVISION))
        parameters = {name: value for name, value in model.named_parameters() if value.requires_grad}
        if not parameters or any("lora_" not in name or "language_model" not in name for name in parameters):
            raise baseline.PreflightError("only_language_lora_parameters_may_train")
        status.update(trainable_parameters=sum(p.numel() for p in parameters.values()),
                      adapted_language_modules=len(matched), adapted_placement=baseline.validate_model_placement(model))

        def adapter_hashes():
            return {name: hashlib.sha256(value.detach().float().cpu().contiguous().numpy().tobytes()).hexdigest()
                    for name, value in get_peft_model_state_dict(model, save_embedding_layers=False).items()}

        initial_hashes = adapter_hashes()

        def batch(key):
            return {name: torch.tensor([value], dtype=torch.long, device="cuda:0")
                    for name, value in prepared[key].items()}

        target_counts = {a["id"]: a["shifted_supervised_tokens"] for a in audits}

        def evaluate():
            model.eval()
            values = []
            with torch.inference_mode():
                for key in validation_ids:
                    loss = float(model(**batch(key), use_cache=False).loss.item())
                    if not math.isfinite(loss):
                        raise baseline.PreflightError("nonfinite_validation_loss")
                    values.append({"id": key, "loss": loss, "supervised_tokens": target_counts[key]})
            return {"loss": sum(v["loss"]*v["supervised_tokens"] for v in values)
                            / sum(v["supervised_tokens"] for v in values), "rows": values}

        phase("base_validation_loss")
        with model.disable_adapter():
            original = heartbeat(evaluate, "Base validation loss")
        baseline.write_json(output_dir / "base_validation_loss.json", original)
        optimizer = torch.optim.AdamW(parameters.values(), lr=5e-5, weight_decay=0.0)
        phase("training", training_called=True)
        for step, microbatch_ids in enumerate(schedule):
            model.train()
            optimizer.zero_grad(set_to_none=True)
            used, losses = [], []
            for micro, key in enumerate(microbatch_ids):
                def forward_backward():
                    loss = model(**batch(key), use_cache=False).loss
                    if not bool(torch.isfinite(loss).all()):
                        raise baseline.PreflightError("nonfinite_training_loss")
                    value = float(loss.detach().item())
                    (loss / len(microbatch_ids)).backward()
                    return value
                losses.append(heartbeat(forward_backward, f"Update {step+1}/{len(schedule)}, microbatch {micro+1}/{len(microbatch_ids)}"))
                used.append(key)
            gradients = [p.grad for p in parameters.values() if p.grad is not None]
            if not gradients or any(not bool(torch.isfinite(g).all()) for g in gradients):
                raise baseline.PreflightError("missing_or_nonfinite_adapter_gradients")
            norm = float(torch.nn.utils.clip_grad_norm_(list(parameters.values()), 1.0, error_if_nonfinite=True))
            if not math.isfinite(norm) or norm <= 0:
                raise baseline.PreflightError("adapter_gradient_norm_not_positive")
            optimizer.step()
            if any(not bool(torch.isfinite(p).all()) for p in parameters.values()):
                raise baseline.PreflightError("nonfinite_adapter_weights")
            status["updates"].append({"step": step+1, "microbatch_ids": used,
                                      "loss": sum(losses)/len(losses), "gradient_norm": norm})
            status["optimizer_steps_completed"] = step+1
            status["training_examples_seen"] += len(used)
            status["memory"] = {"peak_allocated_bytes": torch.cuda.max_memory_allocated(0),
                                "peak_reserved_bytes": torch.cuda.max_memory_reserved(0)}
            phase("training")
            print(f"Completed optimizer update {step+1}/{len(schedule)}; loss={sum(losses)/len(losses):.6f}", flush=True)
        optimizer.zero_grad(set_to_none=True)
        status["epochs_completed"] = 1
        phase("adapted_validation_loss")
        adapted = heartbeat(evaluate, "Adapted validation loss")
        trained_hashes = adapter_hashes()
        if trained_hashes == initial_hashes:
            raise baseline.PreflightError("adapter_weights_unchanged")
        status["adapter_weights_changed"] = True
        phase("saving_adapter")
        adapter_dir = output_dir / "adapter"
        # Vocabulary/embeddings are unchanged and frozen. Avoid the optional
        # automatic Hub config lookup used to detect resized embeddings.
        model.save_pretrained(adapter_dir, safe_serialization=True, save_embedding_layers=False)
        processor.save_pretrained(adapter_dir)
        baseline.write_json(output_dir / "adapter_tensor_hashes.json", trained_hashes)
        # Remove trained adapter modules, then load the files onto the same frozen
        # quantized base. Do not merge or allocate a second 4B base on the T4.
        phase("reloading_adapter")
        del optimizer, parameters, gradients
        bare_model = model.unload()
        del model
        if any("lora_" in name for name, _ in bare_model.named_parameters()):
            raise baseline.PreflightError("adapter_unload_incomplete")
        model = PeftModel.from_pretrained(bare_model, adapter_dir, is_trainable=False)
        if adapter_hashes() != trained_hashes:
            raise baseline.PreflightError("reloaded_adapter_tensor_hash_mismatch")
        status["reloaded_placement"] = baseline.validate_model_placement(model)
        phase("reloaded_validation_loss")
        reloaded = heartbeat(evaluate, "Reloaded validation loss")
        for before, after in zip(adapted["rows"], reloaded["rows"]):
            if before["id"] != after["id"] or not math.isclose(before["loss"], after["loss"], rel_tol=1e-5, abs_tol=1e-5):
                raise baseline.PreflightError("reloaded_validation_loss_mismatch")
        status.update(adapter_reload_verified=True, memory={"gpu": torch.cuda.get_device_name(0),
            "peak_allocated_bytes": torch.cuda.max_memory_allocated(0),
            "peak_reserved_bytes": torch.cuda.max_memory_reserved(0)})
        evaluation = {"purpose": status["purpose"], "model_id": baseline.MODEL_ID,
            "model_revision": smoke.MODEL_REVISION, "dataset_sha256": baseline.DATASET_SHA256,
            "train_rows": 247, "optimizer_steps": len(schedule), "epochs": 1, "validation_ids": validation_ids,
            "test_rows_used": 0, "loss_aggregation": "shifted supervised-token weighted mean across all 21 validation rows",
            "original_validation_loss": original["loss"], "fine_tuned_validation_loss": adapted["loss"],
            "reloaded_validation_loss": reloaded["loss"], "original": original, "fine_tuned": adapted,
            "reloaded": reloaded, "adapter_reload_verified": True,
            "style_improvement_claimed": False, "generations": []}
        # Preserve completed loss evaluation and adapter even if generation later fails.
        evaluation["complete"] = False
        baseline.write_json(output_dir / "evaluation.json", evaluation)
        baseline.write_json(output_dir / "training_log.json", status["updates"])
        phase("validation_generation", saved_generations=0)
        model.eval()
        comparison = {"complete": False, "model_id": baseline.MODEL_ID,
            "model_revision": smoke.MODEL_REVISION, "dataset_sha256": baseline.DATASET_SHA256,
            "epochs": 1, "test_rows_used": 0, "generation": original_baseline["generation"], "results": []}
        save_comparison(output_dir, comparison)

        class FiniteScores(LogitsProcessor):
            def __call__(self, input_ids, scores):
                if bool(torch.isnan(scores).any() | torch.isposinf(scores).any()
                        | ~torch.isfinite(scores).any(dim=-1).all()):
                    raise baseline.PreflightError("invalid_generation_scores")
                return scores

        budget = original_baseline["generation"]["max_new_tokens"]
        for index, (row, old) in enumerate(zip(groups["validation"], original_baseline["results"]), 1):
            phase("validation_generation", saved_generations=index-1, current_id=row["id"])
            inputs = processor.apply_chat_template([baseline.user_message(row)], tokenize=True,
                add_generation_prompt=True, truncation=False, return_dict=True, return_tensors="pt")
            prefix = baseline.input_ids(inputs)
            if (baseline.digest(json.dumps(prefix)) != expected[row["id"]]["prefix_token_ids_sha256"]
                    or len(prefix)+budget > baseline.CONTEXT_BUDGET):
                raise baseline.PreflightError("generation_prefix_or_context_budget_changed")
            inputs = {name: value.to("cuda:0") for name, value in inputs.items()}
            with torch.inference_mode():
                tokens = heartbeat(lambda: model.generate(**inputs, do_sample=False, max_new_tokens=budget,
                    use_cache=True, pad_token_id=processor.tokenizer.pad_token_id, remove_invalid_values=False,
                    logits_processor=LogitsProcessorList([FiniteScores()])), f"Validation generation {index}/21")
            generated = tokens[0, len(prefix):]
            text = processor.tokenizer.decode(generated, skip_special_tokens=True).strip()
            eos = model.generation_config.eos_token_id
            eos = eos if isinstance(eos, list) else [eos]
            result = {"id": row["id"], "source": row["plain_english"], "reference": row["completion"],
                "prompt_sha256": baseline.digest(row["prompt"]), "baseline_output": old["output"], "output": text,
                "output_tokens": int(generated.numel()), "input_tokens": len(prefix),
                "hit_output_limit": generated.numel() >= budget and
                    (not generated.numel() or int(generated[-1]) not in eos),
                "copied_source": " ".join(text.split()) == " ".join(row["plain_english"].split()),
                "exact_reference_match": " ".join(text.split()) == " ".join(row["completion"].split()),
                "source_words": len(row["plain_english"].split()), "output_words": len(text.split()),
                "reference_words": len(row["completion"].split()), "claim_review_status": "pending_review"}
            comparison["results"].append(result)
            save_comparison(output_dir, comparison)
            baseline.validate_result(result)
            print(f"Saved validation generation {index}/21", flush=True)
        comparison["complete"] = True
        evaluation["complete"] = True
        status.update(complete=True, saved_generations=21)
        validate_completion(status, evaluation, comparison, groups, original_baseline)
        baseline.write_json(output_dir / "evaluation.json", evaluation)
        save_comparison(output_dir, comparison)
        save_review(output_dir, comparison)
        status["memory"].update(peak_allocated_bytes=torch.cuda.max_memory_allocated(0),
                                peak_reserved_bytes=torch.cuda.max_memory_reserved(0))
        artifacts = ["evaluation.json", "comparison.json", "validation_generations.jsonl",
                     "prepared_label_audit.json", "training_log.json", "adapter_tensor_hashes.json",
                     "adapter/adapter_config.json", "adapter/adapter_model.safetensors"]
        baseline.write_json(output_dir / "pilot_manifest.json", {"implementation": IMPLEMENTATION,
            "model_revision": smoke.MODEL_REVISION, "dataset_sha256": baseline.DATASET_SHA256,
            "baseline_manifest_sha256": smoke.BASELINE_MANIFEST_SHA256, "smoke_gate_hashes": SMOKE_HASHES,
            "files": {name: hashlib.sha256((output_dir/name).read_bytes()).hexdigest() for name in artifacts}})
        phase("complete")
        print("Pilot complete: one epoch, 247 examples, 124 updates, 21 validation generations", flush=True)
    except Exception as exc:
        failed_phase = status["phase"]
        status.update(complete=False, phase="failed", failed_phase=failed_phase, error_class=type(exc).__name__)
        status.update(baseline.safe_failure_details(exc))
        tb = exc.__traceback__
        while tb is not None:
            if tb.tb_frame.f_code.co_filename == __file__:
                status["stack_locations"].append({"component": "corpus_pilot", "line": tb.tb_lineno})
            tb = tb.tb_next
        if isinstance(exc, baseline.PreflightError):
            status["error_code"] = exc.code
        baseline.write_json(status_path, status)
        raise RuntimeError(f"Pilot stopped in {failed_phase}; inspect pilot_status.json. "
                           "Remote exception messages are suppressed.") from None


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--dataset", required=True)
    parser.add_argument("--baseline-dir", required=True)
    parser.add_argument("--smoke-dir", required=True)
    parser.add_argument("--output-dir", default="/kaggle/working/corpus_v1_pilot")
    parser.add_argument("--check", action="store_true")
    args = parser.parse_args(argv)
    groups, audit, original = load_inputs(args.dataset, args.baseline_dir, args.smoke_dir)
    schedule = epoch_batches([row["id"] for row in groups["train"]])
    print(f"Validated baseline v4 and smoke v1; one epoch, 247 train pairs, {len(schedule)} updates; "
          "21 validation pairs; zero test use")
    if args.check:
        print("Dry-run OK. No credentials accessed, model downloaded or training called.")
        return
    run_training(groups, audit, original, Path(args.output_dir))


if __name__ == "__main__":
    main()


In [ ]:
from pathlib import Path
import subprocess
import sys

root = Path("/kaggle/input")
datasets = list(root.rglob("curated.jsonl"))
baselines = list(root.rglob("baseline_manifest.json"))
smokes = list(root.rglob("smoke_status.json"))
if len(datasets) != 1 or len(baselines) != 1 or len(smokes) != 1:
    raise FileNotFoundError("Attach exactly one corpus Dataset, baseline v4 output, and smoke v1 output")
command = [sys.executable, "/kaggle/working/corpus_pilot.py", "--dataset", str(datasets[0]),
           "--baseline-dir", str(baselines[0].parent), "--smoke-dir", str(smokes[0].parent)]
subprocess.run(command + ["--check"], check=True)
subprocess.run(command, check=True)


In [ ]:
import json
import sys
from pathlib import Path
sys.path.insert(0, "/kaggle/working")
from corpus_pilot import load_inputs, validate_completion
groups, audit, original = load_inputs(datasets[0], baselines[0].parent, smokes[0].parent)
output = Path("/kaggle/working/corpus_v1_pilot")
status = json.loads((output / "pilot_status.json").read_text())
evaluation = json.loads((output / "evaluation.json").read_text())
comparison = json.loads((output / "comparison.json").read_text())
validate_completion(status, evaluation, comparison, groups, original)
print("Verified: one epoch, 247 examples, 124 updates, adapter reload, 21 validation generations.")
print("Original validation loss:", evaluation["original_validation_loss"])
print("Adapted validation loss:", evaluation["fine_tuned_validation_loss"])
print("Review claim preservation and style separately; keep comparison_key.json closed while scoring.")
print("Download the complete corpus_v1_pilot output directory.")
